In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv("../data/loan_data.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 310 entries, 0 to 309
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   age               300 non-null    float64
 1   income            298 non-null    float64
 2   loan_amount       310 non-null    float64
 3   credit_score      302 non-null    float64
 4   employment_years  310 non-null    float64
 5   education         310 non-null    object 
 6   city              310 non-null    object 
 7   approved          310 non-null    int64  
dtypes: float64(5), int64(1), object(2)
memory usage: 19.5+ KB


In [4]:
df.dropna(inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 280 entries, 0 to 309
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   age               280 non-null    float64
 1   income            280 non-null    float64
 2   loan_amount       280 non-null    float64
 3   credit_score      280 non-null    float64
 4   employment_years  280 non-null    float64
 5   education         280 non-null    object 
 6   city              280 non-null    object 
 7   approved          280 non-null    int64  
dtypes: float64(5), int64(1), object(2)
memory usage: 19.7+ KB


In [5]:
df.describe()

,age,income,loan_amount,credit_score,employment_years,approved
count,280.000000,280.000000,280.000000,280.000000,280.000000,280.000000
mean,38.760714,61346.850000,32926.228571,581.539286,9.489286,0.896429
std,10.176043,57921.791113,32232.239386,132.171952,5.605926,0.305249
min,22.000000,20384.000000,5009.000000,5.000000,0.000000,0.000000
25%,29.000000,40156.750000,17133.500000,473.500000,4.000000,1.000000
50%,40.000000,56301.500000,29249.500000,594.500000,10.000000,1.000000
75%,47.000000,77799.000000,45799.000000,686.500000,14.000000,1.000000
max,54.000000,950000.000000,500000.000000,799.000000,19.000000,1.000000


In [6]:
df.head()

,age,income,loan_amount,credit_score,employment_years,education,city,approved
0,50.0,93847.0,42744.0,384.0,10.0,bachelor,Biratnagar,1
1,36.0,99634.0,50543.0,350.0,15.0,high_school,Biratnagar,1
2,29.0,48251.0,47783.0,645.0,3.0,high_school,Kathmandu,1
3,42.0,45945.0,31657.0,669.0,19.0,high_school,Kathmandu,1
4,40.0,52217.0,41187.0,755.0,4.0,high_school,Pokhara,1


In [7]:
# inspect the category/unique values in the 'education' and 'city' columns
print(df["education"].unique())
print(df["city"].unique())
print(df["education"].value_counts())
print(df["city"].value_counts())

['bachelor' 'high_school' 'master']
['Biratnagar' 'Kathmandu' 'Pokhara' 'Lalitpur']
education
bachelor       97
master         92
high_school    91
Name: count, dtype: int64
city
Kathmandu     77
Pokhara       75
Biratnagar    65
Lalitpur      63
Name: count, dtype: int64


In [8]:
# one hot encode the 'education' and 'city' columns
df_ohe = pd.get_dummies(df, columns=["education", "city"])

df_ohe.head()

,age,income,loan_amount,credit_score,employment_years,approved,education_bachelor,education_high_school,education_master,city_Biratnagar,city_Kathmandu,city_Lalitpur,city_Pokhara
0,50.0,93847.0,42744.0,384.0,10.0,1,True,False,False,True,False,False,False
1,36.0,99634.0,50543.0,350.0,15.0,1,False,True,False,True,False,False,False
2,29.0,48251.0,47783.0,645.0,3.0,1,False,True,False,False,True,False,False
3,42.0,45945.0,31657.0,669.0,19.0,1,False,True,False,False,True,False,False
4,40.0,52217.0,41187.0,755.0,4.0,1,False,True,False,False,False,False,True


In [9]:
# label encode the 'education' and 'city' columns
from sklearn.preprocessing import LabelEncoder

df_le = df.copy()

le_edu = LabelEncoder()
le_city = LabelEncoder()

df_le["education_le"] = le_edu.fit_transform(df_le["education"])
df_le["city_le"] = le_city.fit_transform(df_le["city"])

print("Education mapping:", dict(zip(le_edu.classes_, le_edu.transform(le_edu.classes_))))
print("City mapping:", dict(zip(le_city.classes_, le_city.transform(le_city.classes_))))

df_le[["education", "education_le", "city", "city_le"]].head()

Education mapping: {'bachelor': 0, 'high_school': 1, 'master': 2}
City mapping: {'Biratnagar': 0, 'Kathmandu': 1, 'Lalitpur': 2, 'Pokhara': 3}


,education,education_le,city,city_le
0,bachelor,0,Biratnagar,0
1,high_school,1,Biratnagar,0
2,high_school,1,Kathmandu,1
3,high_school,1,Kathmandu,1
4,high_school,1,Pokhara,3


In [14]:
# here lies a problem with label encoding - it introduces an ordinal relationship between the categories which may not be true. 
#{'bachelor': 0, 'high_school': 1, 'master': 2, 'phd': 3}
# this suggests high_school > bachelor which is not true.

# this using manual encoding for the 'education' column
education_mapping = {
    "high_school": 0,
    "bachelor": 1,
    "master": 2,
    "phd": 3
}
df_manual = df.copy()
df_manual["education_manual"] = df_manual["education"].map(education_mapping)
print("Education mapping:", education_mapping)
df_manual[["education", "education_manual"]].sample(10)

Education mapping: {'high_school': 0, 'bachelor': 1, 'master': 2, 'phd': 3}


,education,education_manual
38,high_school,0
171,bachelor,1
127,bachelor,1
201,high_school,0
99,high_school,0
141,high_school,0
275,master,2
208,high_school,0
277,high_school,0
104,master,2


In [11]:
#one hot encode only the 'city' column 
df_final = pd.get_dummies(df_manual, columns=["city"], drop_first=True)

df_final.head()

,age,income,loan_amount,credit_score,employment_years,education,approved,education_manual,city_Kathmandu,city_Lalitpur,city_Pokhara
0,50.0,93847.0,42744.0,384.0,10.0,bachelor,1,1,False,False,False
1,36.0,99634.0,50543.0,350.0,15.0,high_school,1,0,False,False,False
2,29.0,48251.0,47783.0,645.0,3.0,high_school,1,0,True,False,False
3,42.0,45945.0,31657.0,669.0,19.0,high_school,1,0,True,False,False
4,40.0,52217.0,41187.0,755.0,4.0,high_school,1,0,False,False,True


In [12]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 280 entries, 0 to 309
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   age               280 non-null    float64
 1   income            280 non-null    float64
 2   loan_amount       280 non-null    float64
 3   credit_score      280 non-null    float64
 4   employment_years  280 non-null    float64
 5   education         280 non-null    object 
 6   approved          280 non-null    int64  
 7   education_manual  280 non-null    int64  
 8   city_Kathmandu    280 non-null    bool   
 9   city_Lalitpur     280 non-null    bool   
 10  city_Pokhara      280 non-null    bool   
dtypes: bool(3), float64(5), int64(2), object(1)
memory usage: 20.5+ KB
